<a href="https://colab.research.google.com/github/ankit-rathi/Quantvesting_v3/blob/main/notebooks/01_customer_onboarding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Quantvesting | Customer Onboarding

### From one portfolio file → first portfolio assessment

This is the **customer-first entry point**. A new user only needs `Symbol`, `Shares` and `AvgCost`. Account labels and investment history are optional.

**Goal:** get a useful portfolio assessment without asking the customer to understand Quantvesting's internal CSV structure.

## What the customer provides

Minimum CSV:

```text
Symbol,Shares,AvgCost
TCS,100,3200
INFY,150,1450
HDFCBANK,200,1650
```

Accepted common alternatives include `Ticker`, `Quantity` and `Average Price`. If an account column is absent, holdings are treated as `MAIN`.

Investment history is **optional**. It can be added later for XIRR and transaction-based historical performance.

In [ ]:
# 1. Environment
from pathlib import Path
import sys
import pandas as pd

# Point this to the cloned/copied Quantvesting repository.
PROJECT = Path("/content/quantvesting_v3")
if not PROJECT.exists():
    # If you are running from a local Jupyter environment, use:
    # PROJECT = Path(".").resolve()
    PROJECT = Path.cwd()

sys.path.insert(0, str(PROJECT / "src"))

from quantvesting import (
    Quantvesting,
    load_config,
    load_market_data,
    load_portfolio_data,
    onboard_portfolio_csv,
)

print(f"Project: {PROJECT}")

In [ ]:
# 2. Customer identity
# For a beta user, use a simple stable identifier.
PORTFOLIO_ID = "beta_user_001"
PORTFOLIO_DIR = PROJECT / "portfolio_data" / PORTFOLIO_ID
MARKET_DATA_DIR = PROJECT / "market_data"

config = load_config(PROJECT / "config" / "strategy.yaml")
qv = Quantvesting(config)
print(f"Portfolio: {PORTFOLIO_ID}")

## 3. Upload the one required file

In Google Colab, the next cell opens a file picker. In local Jupyter, set `INPUT_FILE` to the path of the CSV you received from the customer.

In [ ]:
# 3. Customer portfolio input
INPUT_FILE = None

try:
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        INPUT_FILE = next(iter(uploaded.keys()))
except ImportError:
    INPUT_FILE = input("Path to customer's portfolio CSV: ").strip()

if not INPUT_FILE:
    raise ValueError("Please provide the customer's portfolio CSV.")

print(f"Input: {INPUT_FILE}")

In [ ]:
# 4. Normalize + persist into the existing engine contract
df_portfolio_input, onboarding_report = onboard_portfolio_csv(
    INPUT_FILE,
    PORTFOLIO_DIR,
)

print("ONBOARDING COMPLETE")
print(f"Input rows:      {onboarding_report['input_rows']}")
print(f"Output rows:     {onboarding_report['output_rows']}")
print(f"Unique securities:{onboarding_report['unique_symbols']}")
print(f"Accounts:        {', '.join(onboarding_report['accounts'])}")
print(f"Saved to:        {onboarding_report['output_path']}")

display(df_portfolio_input.head(10))

In [ ]:
# 5. Load shared Quantvesting data + the newly onboarded portfolio
market_data = load_market_data(MARKET_DATA_DIR)
portfolio_data = load_portfolio_data(
    PORTFOLIO_DIR,
    portfolio_id=PORTFOLIO_ID,
)

# Optional: a customer can add myInvestments.csv later.
print("Shared market data loaded:", {k: len(v) for k, v in market_data.items() if hasattr(v, '__len__')})
print("Portfolio holdings:", len(portfolio_data['portfolio_stocks']))

In [ ]:
# 6. First portfolio assessment
df_portfolio, portfolio_summary = qv.portfolio(
    market_data,
    portfolio_data=portfolio_data,
    eod=False,
    portfolio_id=PORTFOLIO_ID,
)

qv.display_run_summary(portfolio_summary)

print("\nPortfolio coverage")
portfolio_symbols = set(df_portfolio["Symbol"].astype(str))
universe_symbols = set(market_data["prospects"]["Symbol"].astype(str))
outside = sorted(portfolio_symbols - universe_symbols)
print(f"Analysed in Quantvesting universe: {len(portfolio_symbols & universe_symbols)}")
print(f"Outside current universe:          {len(outside)}")
if outside:
    print("Outside-universe holdings:", ", ".join(outside[:20]))

In [ ]:
# 7. Executive terminal
df_prospects = qv.prospects(
    market_data,
    portfolio_data=portfolio_data,
    include_portfolio=True,
    portfolio_id=PORTFOLIO_ID,
)
df_portfolio_actions = qv.portfolio_actions(df_portfolio)
df_prospect_actions = qv.prospect_actions(df_prospects, top_n=10)
df_rotation = qv.capital_rotation(df_prospects, df_portfolio)

qv.display_terminal(
    df_portfolio=df_portfolio,
    df_prospects=df_prospects,
    portfolio_summary=portfolio_summary,
    df_rotation=df_rotation,
    df_portfolio_actions=df_portfolio_actions,
    df_prospect_actions=df_prospect_actions,
    portfolio_id=PORTFOLIO_ID,
    run_id=df_portfolio.attrs.get("run_id"),
)

## What becomes available later

| Customer input | Unlocks |
|---|---|
| Portfolio CSV | Portfolio health, allocation, concentration, P/L, FTT/RRR and existing Quantvesting analytics |
| Investment history | XIRR and transaction-based performance |
| EOD runs over time | Historical portfolio/NAV views |

**Important:** the onboarding layer only normalizes customer input. It does not change the Quantvesting investment methodology.